### Transform Constructors Data
1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case ( constructorld → constructor_1d)
4. Rename columns to make them more meaningful (name → constructor_name )
5. Remove duplicate records
6. Transform values of column nationality to Title Case
7. Write the transformed data to silver constructors table

Step 1 - Loading the env config from the common config

In [0]:
%run ../00-common-Config/01-environment-variable

Step 1 - Read bronze constructors table

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.constructors"
silver_table=f"{catalog_name}.{silver_schema}.constructors"

In [0]:
constructors_df=spark.read.table(bronze_table)

Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as f

In [0]:
constructors_selected_df=constructors_df.select(
    f.col("constructorId"),
    f.col("name"),
    f.col("nationality"),
    f.col("ingestion_timestamp"),
    f.col("source_file")
)

### step 3 && step 4
-  Standardise column names using snake_case ( constructorld → constructor_1d)
-  Rename columns to make them more meaningful (name → constructor_name )

In [0]:
constructors_named_df=(
    constructors_selected_df
    .withColumnRenamed("constructorId","constructor_id")
    .withColumnRenamed("name","constructor_name")
)



### Step 5 Remove duplicate records

In [0]:
constructors_duplicate_df=constructors_named_df.dropDuplicates(["constructor_id"])


### Step 6 Transform values of column nationality to Title Case

In [0]:
constructors_final_df=constructors_duplicate_df.withColumn("nationality",f.initcap(f.col("nationality")))

### Step 7 Write the transformed data to silver constructors table

In [0]:
(
    constructors_final_df
    .write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable(silver_table)
)

In [0]:
spark.read.table(silver_table).display()